<h1>Home Assignment injunctions!</h1>
<li>No Loops. Use only elementwise operations</li>
<li>Your code should work for any data in the provided format. Data values (example, customer ids, product ids, number of customers, etc.) should NEVER appear in your code!</li>

<h1>Problem 1:</h1>
Write a function that reads timeseries pricing data from a file into a pandas dataframe and then groups the data as follows:
<li>The arguments to the function are the filename and a threshold number
<li>The function reads the data in the file and creates a new column "pct_change" with the one day percent change
<li>Then groups the data into four categories:
<ul>
<li>"High+" if the percent change is greater than the threshold 
<li>"Low+" if the percent change is zero or positive and less than or equal to the threshold% 
<li>"Low-" if the percent change is negative but greater than or equal to -1 * the threshold
<li>"High-" if the percent change is less than -1 * the threshold
</ul>
<li>The function should return a dataframe that contains three columns (count, mean, stdev) and four index values (High+, High-, Low+, Low-)
    <p><b>Note: </b>we have to deal with nan percent changes. Make sure that you don't count a NaN in any of the four categories! (see https://pandas.pydata.org/docs/reference/api/pandas.isna.html)  <p>
For the sample data your function should return a dataframe with the following values for a threshold of 1.0:

<pre>
        count	mean	stdev
High+	870	2.162834	1.301745
High-	666	-2.291770	1.370657
Low+	870	0.463122	0.284869
Low-	883	-0.445557	0.276355

</pre>

    

<h3>read_csv</h3>
The pandas <a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html">read_csv</a> function reads data from a delimited file into a pandas dataframe.



<h2>Sample data</h2>
Use the attached AAPL.csv file

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def change_groups(datafile,threshold):
    #Your code goes here
    
    #read the data in the file
    df = pd.read_csv(datafile)
    
    #calculate the one day percent change
    p_change = df.iloc[:,1].pct_change()
    p_change = p_change.to_frame()
    p_change.columns = ["pct"]
    p_change["pct"] = p_change["pct"]*100
    
    #group the data into four categories
    p_change["cat"] = p_change.apply(lambda x: 
                                     "High+" if x.pct > threshold
                                     else "High-" if x.pct < -1*threshold
                                     else "Low+" if x.pct >= 0 and x.pct <= threshold
                                     else "Low-" if x.pct < 0 and x.pct >= -1*threshold
                                     else "NaN",
                                     axis = 1)
    
    #analysis by four categories
    output = p_change.groupby(by = "cat").count()
    output.columns = ["count"]
    output["mean"] = p_change.groupby(by = "cat").mean()
    output["stdev"] = p_change.groupby(by = "cat").std()
    output.index.name = None
    
    print(output[0:4])

In [ ]:
df = pd.read_csv("AAPL.csv")
p_change = df.iloc[:,1].pct_change()
p_change = p_change.to_frame()
p_change.columns = ["pct"]
p_change["pct"] = p_change["pct"]*100

threshold=1.0
p_change["cat"] = p_change.apply(lambda x: 
                                 "High+" if x.pct > threshold
                                 else "High-" if x.pct < -1*threshold
                                 else "Low+" if x.pct >= 0 and x.pct <= threshold
                                 else "Low-" if x.pct < 0 and x.pct >= -1*threshold
                                 else "NaN",
                                 axis = 1)

p_change

In [ ]:
#Test your code
change_groups("AAPL.csv",1.0)

<h1>Problem 2</h1>
A manufacturer has data on orders from customers and product prices in two dataframes (see below). They want to use this data to answer the following questions:
<ol>
    <li>Which customer is responsible for the most revenue</li>
    <li>Which customer is responsible for the highest profit</li>
    <li>Which product is responsible for the highest (dollar) profit</li>
    <li>Which customer and product combination is responsible for the most orders</li>
</ol>
<p>
Obviously, your code should work for any actual data values and pandas dataframes of any length!
<p>For the data below: your answers should be:
    
<pre>
Customer with most profit: 005
Customer with most revenue: 007
Product with most profit: 011
Customer 001 with product 010 with 4 orders is the most ordered customer product pair
</pre>


<h2>Useful functions:</h2>
<li><a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.sort_values.html">pd.sort_values</a> </li>
<li><a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.join.html">pandas dataframe join</a> the last example on the linked page is probably what you need here!</li>
<li><a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html">pd.groupby</a></li>

In [ ]:
import pandas as pd
import numpy as np
order_data = pd.DataFrame([["001","010",24],
                           ["007","012",35],
                           ["001","011",15],
                           ["005","010",30],
                           ["007","011",17],
                           ["005","011",81],
                           ["001","010",32],
                           ["007","012",89],
                           ["001","010",16],
                           ["001","010",33]],columns=["customer","product","amount"])
products = pd.DataFrame([['010',22.3,17.2],
                        ['011',11.7,5.5],
                        ['012',62.5,61.4]],columns=['product','price','cost'])                     

In [ ]:
order_data

In [ ]:
products

In [ ]:
#Your code goes here

#join order_data and products into one dataframe
##and calculate revenue and profit 
data = order_data.join(products.set_index("product"), on="product")
data = data.assign(revenue = data.amount * data.price)
data = data.assign(profit = data.amount * (data.price - data.cost))
data

In [ ]:
#check which customer is responsible for the most revenue and highest profit
customer_value = data.groupby(by = "customer").sum(numeric_only = True).sort_values(by = "revenue")
customer_value

In [ ]:
#check which product is responsible for the highest (dollar) profit
product_profit = data.groupby(by = "product").sum(numeric_only = True).sort_values(by = "profit")
product_profit

In [ ]:
#check which customer and product combination is responsible for the most order
pair_frequency = data.groupby(["customer","product"]).count().sort_values(by = "amount")
pair_frequency

In [ ]:
#print the answer:
print("Customer with most profit: " + customer_value["profit"].idxmax())
print("Customer with most revenue: " + customer_value["revenue"].idxmax())
print("Product with most profit: " + product_profit["profit"].idxmax())
print("Customer " + pair_frequency["amount"].idxmax()[0] 
      + " with product " + pair_frequency["amount"].idxmax()[1] 
      +  " with " + str(pair_frequency["amount"].max()) 
      + " orders is the most ordered customer product pair")